# U-Net From Scratch untuk FloPWD 2025

Notebook ini memakai U-Net murni dari `torch.nn`, tanpa encoder pretrained, tanpa ImageNet weights, tanpa Hugging Face, dan tanpa `segmentation_models_pytorch`.

Aktifkan GPU di Colab melalui `Runtime > Change runtime type > GPU`, lalu jalankan cell dari atas ke bawah.

Struktur dataset default:

```text
/content/drive/MyDrive/semantix/data/
  Raw_Images/
  Segmentation_Masks/
  Image_labels_Binary Classification Task.csv
  Mask_foreground_percentages_Regression Task.csv
```


In [ ]:
# 1. Install dependencies
%pip install -q albumentations opencv-python-headless scikit-learn pandas matplotlib tqdm


In [ ]:
# 2. Imports & reproducibility
import os
import random
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.autonotebook import tqdm

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f'Using device: {DEVICE.upper()}')


In [ ]:
# 3. Mount Google Drive
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ModuleNotFoundError:
    print("Bukan environment Colab; skip mount Google Drive.")


In [ ]:
# 4. Configuration
# Ubah PROJECT_DIR kalau folder Drive kamu berbeda.
PROJECT_DIR = "/content/drive/MyDrive/semantix"
DATA_DIR = os.path.join(PROJECT_DIR, "data")
IMAGE_DIR = os.path.join(DATA_DIR, "Raw_Images")
MASK_DIR = os.path.join(DATA_DIR, "Segmentation_Masks")
LABEL_CSV = os.path.join(DATA_DIR, "Image_labels_Binary Classification Task.csv")
FOREGROUND_CSV = os.path.join(DATA_DIR, "Mask_foreground_percentages_Regression Task.csv")
MODEL_SAVE_DIR = os.path.join(PROJECT_DIR, "model")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

IMAGE_HEIGHT = 512
IMAGE_WIDTH = 512
BATCH_SIZE = 4      # Turunkan ke 2 atau 1 kalau GPU Colab OOM.
EPOCHS = 30
PATIENCE = 8
LR = 1e-4
WEIGHT_DECAY = 1e-4
BASE_CHANNELS = 32  # Ubah ke 64 kalau GPU kuat dan ingin model lebih besar.

BEST_MODEL_PATH = os.path.join(MODEL_SAVE_DIR, "best_flopwd_unet_from_scratch.pth")
HISTORY_CSV_PATH = os.path.join(MODEL_SAVE_DIR, "unet_from_scratch_history.csv")

print("DATA_DIR       :", DATA_DIR)
print("IMAGE_DIR      :", IMAGE_DIR)
print("MASK_DIR       :", MASK_DIR)
print("MODEL_SAVE_DIR :", MODEL_SAVE_DIR)

required_paths = [DATA_DIR, IMAGE_DIR, MASK_DIR, LABEL_CSV, FOREGROUND_CSV]
missing_paths = [path for path in required_paths if not os.path.exists(path)]
if missing_paths:
    print("\nPath berikut belum ditemukan. Sesuaikan PROJECT_DIR/DATA_DIR atau upload data ke Drive:")
    for path in missing_paths:
        print("-", path)
else:
    print("\nSemua path dataset ditemukan.")


In [ ]:
# 5. Load dataset metadata
label_df = pd.read_csv(LABEL_CSV)
foreground_df = pd.read_csv(FOREGROUND_CSV)

label_df.columns = label_df.columns.str.strip()
foreground_df.columns = foreground_df.columns.str.strip()

label_df = label_df.rename(columns={
    label_df.columns[0]: "image_name",
    label_df.columns[1]: "has_plastic",
})
foreground_df = foreground_df.rename(columns={
    foreground_df.columns[0]: "image_name",
    foreground_df.columns[1]: "foreground_pct",
})

df = pd.merge(label_df, foreground_df, on="image_name", how="inner")

def build_image_path(name):
    base = os.path.splitext(name)[0]
    return os.path.join(IMAGE_DIR, base + ".jpg")

def build_mask_path(name):
    base = os.path.splitext(name)[0]
    return os.path.join(MASK_DIR, base + "_mask.png")

df["image_path"] = df["image_name"].apply(build_image_path)
df["mask_path"] = df["image_name"].apply(build_mask_path)

df["img_exists"] = df["image_path"].apply(os.path.exists)
df["mask_exists"] = df["mask_path"].apply(os.path.exists)
df = df[df["img_exists"] & df["mask_exists"]].reset_index(drop=True)

df["has_plastic"] = df["has_plastic"].astype(int)

print(f"Total valid samples: {len(df)}")
print(f"With plastic       : {df['has_plastic'].sum()}")
print(f"Without plastic    : {(df['has_plastic'] == 0).sum()}")
df.head()


In [ ]:
# 6. Split data 70:15:15, stratified by has_plastic
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    stratify=df["has_plastic"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["has_plastic"],
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {len(train_df)}")
print(f"Val  : {len(val_df)}")
print(f"Test : {len(test_df)}")


In [ ]:
# 7. Augmentation
# Normalize(mean=0, std=1) hanya mengubah pixel 0..255 menjadi 0..1.
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(shift_limit=0.1, scale_limit=0.2, rotate_limit=30, p=0.5),
    A.OneOf([
        A.ElasticTransform(alpha=1, sigma=50, p=0.5),
        A.GridDistortion(p=0.5),
    ], p=0.3),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
    A.GaussNoise(std_range=(0.01, 0.05), p=0.3),
    A.CoarseDropout(
        num_holes_range=(1, 8),
        hole_height_range=(16, 32),
        hole_width_range=(16, 32),
        p=0.3,
    ),
    A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
    A.Normalize(mean=(0, 0, 0), std=(1, 1, 1), max_pixel_value=255.0),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Resize(IMAGE_HEIGHT, IMAGE_WIDTH),
    A.Normalize(mean=(0, 0, 0), std=(1, 1, 1), max_pixel_value=255.0),
    ToTensorV2(),
])


In [ ]:
# 8. Dataset & DataLoader
class FloPWDDataset(Dataset):
    def __init__(self, dataframe, transforms=None):
        self.dataframe = dataframe
        self.transforms = transforms

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image = cv2.imread(row["image_path"])
        if image is None:
            raise FileNotFoundError(row["image_path"])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise FileNotFoundError(row["mask_path"])
        mask = (mask > 127).astype(np.float32)

        if self.transforms:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        mask = mask.unsqueeze(0).float()  # [1, H, W] untuk BCEWithLogitsLoss
        return image.float(), mask

train_dataset = FloPWDDataset(train_df, transforms=train_transform)
val_dataset = FloPWDDataset(val_df, transforms=val_transform)
test_dataset = FloPWDDataset(test_df, transforms=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")


In [ ]:
# 9. U-Net architecture from scratch
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class Down(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels),
        )

    def forward(self, x):
        return self.block(x)

class Up(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x, skip):
        x = self.up(x)

        diff_y = skip.size(2) - x.size(2)
        diff_x = skip.size(3) - x.size(3)
        if diff_y != 0 or diff_x != 0:
            x = F.pad(x, [diff_x // 2, diff_x - diff_x // 2, diff_y // 2, diff_y - diff_y // 2])

        x = torch.cat([skip, x], dim=1)
        return self.conv(x)

class UNetFromScratch(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, base_channels=32):
        super().__init__()
        c1 = base_channels
        c2 = base_channels * 2
        c3 = base_channels * 4
        c4 = base_channels * 8
        c5 = base_channels * 16

        self.inc = DoubleConv(in_channels, c1)
        self.down1 = Down(c1, c2)
        self.down2 = Down(c2, c3)
        self.down3 = Down(c3, c4)
        self.down4 = Down(c4, c5)
        self.up1 = Up(c5, c4, c4)
        self.up2 = Up(c4, c3, c3)
        self.up3 = Up(c3, c2, c2)
        self.up4 = Up(c2, c1, c1)
        self.outc = nn.Conv2d(c1, out_channels, kernel_size=1)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        return self.outc(x)

model = UNetFromScratch(in_channels=3, out_channels=1, base_channels=BASE_CHANNELS).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model.__class__.__name__)
print(f"Total parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("Pretrained weights  : None")


In [ ]:
# 10. Loss function
# Estimate positive pixel weight untuk membantu class imbalance.
pos_pixels = 0
neg_pixels = 0
subset = train_df.sample(min(200, len(train_df)), random_state=SEED)

for _, row in subset.iterrows():
    mask = cv2.imread(row["mask_path"], cv2.IMREAD_GRAYSCALE)
    mask = (mask > 127).astype(np.uint8)
    pos_pixels += int(mask.sum())
    neg_pixels += int(mask.size - mask.sum())

pos_weight_value = neg_pixels / max(pos_pixels, 1)
pos_weight_value = min(float(pos_weight_value), 50.0)
print(f"Estimated pos_weight: {pos_weight_value:.2f}")

bce_loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], device=DEVICE))

def dice_loss_fn(logits, targets, smooth=1.0):
    probs = torch.sigmoid(logits)
    intersection = (probs * targets).sum(dim=(1, 2, 3))
    union = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2.0 * intersection + smooth) / (union + smooth)
    return 1.0 - dice.mean()

def combined_loss(logits, targets, dice_w=0.7, bce_w=0.3):
    d_loss = dice_loss_fn(logits, targets)
    b_loss = bce_loss_fn(logits, targets)
    return dice_w * d_loss + bce_w * b_loss


In [ ]:
# 11. Optimizer, scaler, metrics
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

def compute_stats(preds, targets, eps=1e-7):
    preds = preds.float()
    targets = targets.float()

    tp = (preds * targets).sum(dim=(1, 2, 3))
    fp = (preds * (1 - targets)).sum(dim=(1, 2, 3))
    fn = ((1 - preds) * targets).sum(dim=(1, 2, 3))

    iou = (tp + eps) / (tp + fp + fn + eps)
    dice = (2 * tp + eps) / (2 * tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)

    return {
        "iou": iou.mean().item(),
        "dice": dice.mean().item(),
        "precision": precision.mean().item(),
        "recall": recall.mean().item(),
    }


In [ ]:
# 12. Train, validation, threshold search

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    pbar = tqdm(loader, desc="Training", leave=False)

    for images, masks in pbar:
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            logits = model(images)
            loss = combined_loss(logits, masks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        pbar.set_postfix(loss=loss.item())

    return running_loss / max(len(loader), 1)


def validate_one_epoch(model, loader, device, threshold=0.5):
    model.eval()
    running_loss = 0.0
    iou_scores, dice_scores, precision_scores, recall_scores = [], [], [], []

    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Validation", leave=False):
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True)

            logits = model(images)
            loss = combined_loss(logits, masks)

            probs = torch.sigmoid(logits)
            preds = (probs > threshold).float()
            stats = compute_stats(preds, masks)

            running_loss += loss.item()
            iou_scores.append(stats["iou"])
            dice_scores.append(stats["dice"])
            precision_scores.append(stats["precision"])
            recall_scores.append(stats["recall"])

    return {
        "loss": running_loss / max(len(loader), 1),
        "iou": float(np.mean(iou_scores)),
        "dice": float(np.mean(dice_scores)),
        "precision": float(np.mean(precision_scores)),
        "recall": float(np.mean(recall_scores)),
    }


def find_best_threshold(model, loader, device):
    thresholds = np.arange(0.10, 0.91, 0.05)
    best_threshold = 0.5
    best_dice = -1.0
    all_probs, all_masks = [], []

    model.eval()
    with torch.no_grad():
        for images, masks in tqdm(loader, desc="Collecting val probs", leave=False):
            images = images.to(device, non_blocking=True)
            logits = model(images)
            probs = torch.sigmoid(logits).cpu()
            all_probs.append(probs)
            all_masks.append(masks.cpu())

    all_probs = torch.cat(all_probs, dim=0)
    all_masks = torch.cat(all_masks, dim=0)

    for threshold in thresholds:
        preds = (all_probs > threshold).float()
        stats = compute_stats(preds, all_masks)
        if stats["dice"] > best_dice:
            best_dice = stats["dice"]
            best_threshold = float(threshold)

    return best_threshold, best_dice


In [ ]:
# 13. Training loop with early stopping
best_val_dice = -1.0
history = []
patience_counter = 0

print(f"Starting training for {EPOCHS} epochs...")
print(f"Saving best model to: {BEST_MODEL_PATH}")

for epoch in range(EPOCHS):
    print(f"\nEpoch [{epoch + 1}/{EPOCHS}]")

    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    best_thr, _ = find_best_threshold(model, val_loader, DEVICE)
    val_metrics = validate_one_epoch(model, val_loader, DEVICE, threshold=best_thr)

    print(f"Train Loss   : {train_loss:.4f}")
    print(f"Val Loss     : {val_metrics['loss']:.4f}")
    print(f"Val IoU      : {val_metrics['iou']:.4f}")
    print(f"Val Dice     : {val_metrics['dice']:.4f}")
    print(f"Val Precision: {val_metrics['precision']:.4f}")
    print(f"Val Recall   : {val_metrics['recall']:.4f}")
    print(f"Best Threshold: {best_thr:.2f}")

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        **{f"val_{k}": v for k, v in val_metrics.items()},
        "best_threshold": best_thr,
    })
    pd.DataFrame(history).to_csv(HISTORY_CSV_PATH, index=False)

    if val_metrics["dice"] > best_val_dice:
        best_val_dice = val_metrics["dice"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "base_channels": BASE_CHANNELS,
            "best_threshold": best_thr,
            "best_val_dice": best_val_dice,
        }, BEST_MODEL_PATH)
        print("Best model saved.")
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"No improvement. Patience: {patience_counter}/{PATIENCE}")
        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break

print("\nTraining complete.")
print(f"Best Val Dice: {best_val_dice:.4f}")


In [ ]:
# 14. Test evaluation
checkpoint = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
best_thr = float(checkpoint.get("best_threshold", 0.5))

print(f"Using threshold: {best_thr:.2f}")
test_metrics = validate_one_epoch(model, test_loader, DEVICE, threshold=best_thr)

print("\nTest Results")
print(f"Loss     : {test_metrics['loss']:.4f}")
print(f"IoU      : {test_metrics['iou']:.4f}")
print(f"Dice     : {test_metrics['dice']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall   : {test_metrics['recall']:.4f}")


In [ ]:
# 15. Training history plot
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(hist_df["epoch"], hist_df["train_loss"], label="Train Loss")
axes[0].plot(hist_df["epoch"], hist_df["val_loss"], label="Val Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("Loss Curve")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(hist_df["epoch"], hist_df["val_iou"], label="Val IoU")
axes[1].plot(hist_df["epoch"], hist_df["val_dice"], label="Val Dice")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Score")
axes[1].set_title("IoU & Dice")
axes[1].legend()
axes[1].grid(True)

axes[2].plot(hist_df["epoch"], hist_df["val_precision"], label="Val Precision")
axes[2].plot(hist_df["epoch"], hist_df["val_recall"], label="Val Recall")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Score")
axes[2].set_title("Precision & Recall")
axes[2].legend()
axes[2].grid(True)

plt.tight_layout()
history_plot_path = os.path.join(MODEL_SAVE_DIR, "unet_from_scratch_training_history.png")
plt.savefig(history_plot_path, dpi=150)
plt.show()
print("Saved:", history_plot_path)


In [ ]:
# 16. Sample prediction visualization

def visualize_predictions(model, dataset, device, threshold=0.5, num_samples=6):
    model.eval()
    num_samples = min(num_samples, len(dataset))
    fig, axes = plt.subplots(num_samples, 3, figsize=(15, 5 * num_samples))
    if num_samples == 1:
        axes = np.expand_dims(axes, axis=0)

    indices = np.random.choice(len(dataset), num_samples, replace=False)

    for row_idx, dataset_idx in enumerate(indices):
        image, mask = dataset[dataset_idx]
        image_tensor = image.unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(image_tensor)
            prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
            pred = (prob > threshold).astype(np.float32)

        img_np = image.permute(1, 2, 0).cpu().numpy()
        img_np = np.clip(img_np, 0, 1)
        mask_np = mask[0].cpu().numpy().astype(np.float32)

        axes[row_idx, 0].imshow(img_np)
        axes[row_idx, 0].set_title("Input Image")
        axes[row_idx, 0].axis("off")

        axes[row_idx, 1].imshow(mask_np, cmap="gray", vmin=0, vmax=1)
        axes[row_idx, 1].set_title("Ground Truth")
        axes[row_idx, 1].axis("off")

        axes[row_idx, 2].imshow(pred, cmap="gray", vmin=0, vmax=1)
        axes[row_idx, 2].set_title(f"Prediction (thr={threshold:.2f})")
        axes[row_idx, 2].axis("off")

    plt.tight_layout()
    pred_path = os.path.join(MODEL_SAVE_DIR, "unet_from_scratch_predictions.png")
    plt.savefig(pred_path, dpi=150)
    plt.show()
    print("Saved:", pred_path)

visualize_predictions(model, test_dataset, DEVICE, threshold=best_thr, num_samples=6)
